# Análise exploratória dos dados
A seguir, vamos fazer a parte da EDA do projeto. Específicamente da tabela `alunos_alfabetizacao` da camada Gold que foi construída no tech challenge da fase 2.

## Pergunta norteadora
Dado o contexto do aluno, da escola e do município, é possível prever se um aluno será considerado alfabetizado?

## Variável Target
Nossa variável target será a coluna `alfabetizado`, respondendo a pergunta norteadora se um aluno será considerado alfabetizado ou não.

In [ ]:
# Instalação do matplotlib e seaborn para importar as bibliotecas
!pip install matplotlib seaborn

In [ ]:
# Importações
# Utilizaremos pandas para ler o arquivo de dados
import pandas as pd

# Matplotlib e Seaborn para podermos plotar os gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Para separar em treino e teste vamos utilizar train_test_split
from sklearn.model_selection import train_test_split

# Utilizaremos o standard de Scaler
from sklearn.preprocessing import StandardScaler


# Ávore de decisão
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Carregando os dados, que são provenientes da fase 2
df_alfabetizacao = pd.read_parquet("../data/gold/alunos_alfabetizacao.parquet")

In [ ]:
df_alfabetizacao.head()

In [ ]:
df_alfabetizacao.describe().round(2)

In [ ]:
df_alfabetizacao.dtypes

In [ ]:
df_alfabetizacao.isnull().sum()

In [ ]:
# Exibindo total de alunos alfabetizados, separados por sim/não
print(f"Quantidade da coluna {df_alfabetizacao["alfabetizado"].value_counts()}")
print(f"Proporção da coluna {(df_alfabetizacao["alfabetizado"].value_counts(normalize=True) * 100).round(2)}")

In [ ]:
for colunas_categoricas in ["id_municipio_nome","sigla_uf","sigla_uf_nome", "alfabetizado", "serie", "rede", "presenca", "preenchimento_caderno"]:
    print(f"Coluna: {df_alfabetizacao[colunas_categoricas].value_counts()}")
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

In [ ]:
todas_ufs = {"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}
ufs_presentes = set(df_alfabetizacao["sigla_uf"].dropna().unique())

print("UFs faltando:", todas_ufs - ufs_presentes)

**Limitação**: faltam 4 UFs inteiras na base: SP, AC, RR e DF. </br>
SP é a mais grave, por ser o estado mais populoso do Brasil, qualquer modelo treinado aqui não pode ser generalizado para esse estado.

In [ ]:
# Identificando outlier presente na coluna "peso_aluno", confirmando que é acima de 0.999
print(df_alfabetizacao["peso_aluno"].quantile([0.95, 0.99, 0.999, 1.0]))

In [ ]:
# Descobrindo de quais UFs estão vindo os outliers
df_alfabetizacao.nlargest(10, "peso_aluno")[["id_aluno","sigla_uf","id_municipio_nome","rede","peso_aluno"]]

In [ ]:
# Todos os alunos que tem presença sim/não são alfabetizados?
print("Comparativo alfabetizado vs presença")
print(pd.crosstab(df_alfabetizacao["presenca"], df_alfabetizacao["alfabetizado"]))

## Decisão de base
Após comparar quantos dos alunos tinham a coluna preenchida com alfabetizado sim/não observamos que todos que estavam "Ausentes" foram classificados como "Não", essa é uma **regra de atribuição do sistema de avaliação**. Sem nota registrada quer dizer que não passa no corte, não uma medição real da alfabetização desses alunos.</br></br>
Decidimos que não vamos incluir os alunos que foram classificados como "Ausentes" pois, na nossa base, não temos agregações que ajudem a descobrir a causa real do porquê os alunos não puderam realizar a prova.

In [ ]:
df_alfabetizacao_presentes = df_alfabetizacao[df_alfabetizacao["presenca"] == "Presente"].copy()
qtde_anterior = len(df_alfabetizacao)
qtde_atual = len(df_alfabetizacao_presentes)

print(f"Quantidade de alunos antes do filtro de presentes: {qtde_anterior}")
print(f"Quantidade de alunos após: {qtde_atual}")
print(f"Removidos {qtde_anterior - qtde_atual} ({(qtde_anterior-qtde_atual)/qtde_anterior*100:.2f}%)")

In [ ]:
df_alfabetizacao_presentes.isnull().sum()

Podemos observar que tanto a quantidade de nulos da coluna peso_aluno quanto da proficiência diminuíram exatamente na mesma quantidade de alunos que identificamos como "Ausentes" (de 244.630 para 249) confirmando que o filtro de presença resolveu, por consequência, o problema de nulo nessas duas colunas. Os 249 casos residuais são alunos marcados como "Presente" que, mesmo assim, não completaram a prova (prova em branco/invalidada).

In [ ]:
# Check de proficiencia dos casos residuais
print(pd.crosstab(df_alfabetizacao_presentes["preenchimento_caderno"], df_alfabetizacao_presentes["proficiencia"].isna()))

### DATA LEAKAGE
Identificamos que as colunas a seguir contém dados que já dariam a resposta ao modelo.

- proficiencia </br>
    A target é calculada a partir dessa variável (que vimos na fase 2 sobre a nota de corte: 743)
- taxa_alfabetizacao_municipio </br>
    É a porcentagem de alunos alfabetizados por município, o calculo inclui o aluno da linha
- taxa_alfabetizacao_uf </br>
    Mesmo mecanismo da coluna acima, só que diluído por UF (grupo maior → vazamento mais fraco)
- media_portugues_municipio </br>
    É a média de proficiência do município. Mesmo problema de taxa_alfabetizacao_municipio, incluindo a nota do próprio aluno no cálculo
- media_portugues_uf </br>
    Mesmo mecanismo da coluna "media_portugues_municipio", diluído por UF

In [ ]:
# Identificando leakage da coluna proficiencia
df_leakage = df_alfabetizacao_presentes.dropna(subset=["proficiencia"]).copy()
df_leakage["regra_aplicada"] = (df_leakage["proficiencia"] >= 743).map({True: "Sim", False: "Não"})
regra = (df_leakage["regra_aplicada"] == df_leakage["alfabetizado"]).mean() * 100
print(f"Porcentagem de vezes que 'proficiencia >= 743' bate com o alfabetizado real: {regra:.2f}%")

In [ ]:
colunas_leakage = [
    "proficiencia",
    "taxa_alfabetizacao_municipio",
    "taxa_alfabetizacao_uf",
    "media_portugues_municipio",
    "media_portugues_uf",
]

df_modelagem = df_alfabetizacao_presentes.drop(columns=colunas_leakage)
print(df_modelagem.shape)

In [ ]:
colunas_pairplot = [
    "dsu_ef_anos_iniciais", "inse_medio", "meta_alfabetizacao_2024_municipio",
    "meta_alfabetizacao_2024_uf", "pib_per_capita", "alfabetizado"
]

amostra = df_modelagem[colunas_pairplot].sample(3000, random_state=42)
sns.pairplot(amostra, hue="alfabetizado")

## Correlação

In [ ]:
df_modelagem["alfabetizado_bin"] = (df_modelagem["alfabetizado"] == "Sim").astype(int)

correlacao = df_modelagem.corr(numeric_only=True)

plt.figure(figsize=(20,10))
sns.heatmap(correlacao, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlação entre colunas")
plt.show()

Vamos remover os IDs que não fazem parte da análise na correlação

In [ ]:
colunas_remover_correlacao = ["ano", "id_aluno", "id_escola", "id_municipio"]

correlacao_limpa = df_modelagem.drop(columns=colunas_remover_correlacao).corr(numeric_only=True)

plt.figure(figsize=(18,9))
sns.heatmap(correlacao_limpa, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlação entre as colunas (sem IDs)")
plt.show()

### Multicolinearidade 1: população / pib / quantidade_escolas_inse

`população`, `pib` e `quantidade_escolas_inse` estão medindo "tamanho do município" de formas diferentes.

- `pib` (bruto) é descartado por ser redundante com `pib_per_capita`, que já usamos e que não sofre desse mesmo problema (correlaciona só 0,09-0,19 com esse cluster, porque mede riqueza por pessoa, não tamanho).
- `quantidade_escolas_inse` Correlaciona bem com população porque cidade grande normalmente tem mais escolas participando do SAEB, não porque a coluna "signifique" tamanho de cidade.

Vamos manter: `populacao` e `pib_per_capita`.

In [ ]:
colunas_multico = ["pib", "quantidade_escolas_inse"]
df_modelagem = df_modelagem.drop(columns=colunas_multico)
print(df_modelagem.shape)

### Multicolinearidade 2: taxa_aprovacao / taxa_reprovacao_ef_2_ano_anterior

`taxa_aprovacao_ef_2_ano_anterior` e `taxa_reprovacao_ef_2_ano_anterior` correlacionam -0,99 entre si, é redundância (aprovação + reprovação + abandono ≈ 100%). Mantemos `taxa_aprovacao_ef_2_ano_anterior` (correlação mais forte com a target: 0,051 vs. -0,030 da reprovação) e descartamos `taxa_reprovacao_ef_2_ano_anterior`. `taxa_abandono_ef_2_ano_anterior` não faz parte dessa redundância, é mantida por ter a correlação mais forte entre as três (-0,130).

In [ ]:
colunas_multico_2 = ["taxa_reprovacao_ef_2_ano_anterior"]
df_modelagem = df_modelagem.drop(columns=colunas_multico_2)
print(df_modelagem.shape)

### Multicolinearidade 3: icg_nivel_1 a icg_nivel_6

Os 6 níveis do ICG (Complexidade de Gestão da Escola) correlacionam fortemente entre si (0,62 a 0,92), mas nenhum individualmente correlaciona com a target (todos abaixo de 0,03). 

- **Somar os níveis**: descartado, porque os valores não somam ~100% por município (não são percentual do total de escolas), então a soma não teria interpretação clara.

Decisão: descartar os 6 níveis do ICG por completo.

In [ ]:
colunas_multico_3 = ["icg_nivel_1", "icg_nivel_2", "icg_nivel_3", "icg_nivel_4", "icg_nivel_5", "icg_nivel_6"]
df_modelagem = df_modelagem.drop(columns=colunas_multico_3)
print(df_modelagem.shape)

### had_ef_anos_iniciais

HAD = Horas-Aula Diária. É a quantidade média de horas por dia que os alunos passam em aula, no município, considerando as escolas dos anos iniciais do Ensino Fundamental.

In [ ]:
total_municipios = df_modelagem["id_municipio"].nunique()
municipios_sem_had = df_modelagem[df_modelagem["had_ef_anos_iniciais"].isna()]["id_municipio"].nunique()

print(f"Total de municípios: {total_municipios}")
print(f"Municípios com pelo menos 1 aluno sem had_ef_anos_iniciais: {municipios_sem_had}")

In [ ]:
por_municipio = df_modelagem.groupby("id_municipio")["had_ef_anos_iniciais"].apply(lambda x: x.isna().mean())
print(por_municipio.value_counts(bins=[-0.01, 0.01, 0.5, 0.99, 1.0]))

In [ ]:
municipios = df_modelagem.drop_duplicates("id_municipio")[["id_municipio", "populacao", "had_ef_anos_iniciais"]].copy()
municipios["tem_had"] = municipios["had_ef_anos_iniciais"].notna()

print(municipios.groupby("tem_had")["populacao"].describe())

`had_ef_anos_iniciais` 1.053 municípios com 100% nulo, 3.818 com 100% preenchido. Sugere que municípios pequenos não atingem um critério mínimo do INEP para esse cálculo.

Levantando o dado de que a correlação com a target também é pequena, vamos remover do dataset

In [ ]:
df_modelagem = df_modelagem.drop(columns=["had_ef_anos_iniciais"])
print(df_modelagem.shape)

### peso_aluno

No heatmap, `peso_aluno` não correlaciona com quase nada, não ajuda a explicar a alfabetização

Se parece com o outlier que já investigamos (máximo 142,55, mais ou menos 31x o percentil 99,9%, concentrado no Rio de Janeiro) reforça ainda mais que essa coluna tem uma lógica própria de desenho amostral, não uma relação direta com o desempenho do aluno

Vamos descartar `peso_aluno` também

In [ ]:
df_modelagem = df_modelagem.drop(columns=["peso_aluno"])
print(df_modelagem.shape)

In [ ]:
colunas_nulos = [
    "dsu_ef_anos_iniciais", "afd_ef_anos_iniciais_grupo_1", "ird_alta", "ird_baixa_regularidade",
    "tdi_ef_2_ano_anterior", "taxa_aprovacao_ef_2_ano_anterior", "taxa_abandono_ef_2_ano_anterior",
    "inse_medio",
]

antes = len(df_modelagem)
linhas_com_algum_nulo = df_modelagem[colunas_nulos].isna().any(axis=1).sum()

print(f"Total de alunos: {antes}")
print(f"Alunos com nulo em pelo menos uma das 8 colunas: {linhas_com_algum_nulo} ({linhas_com_algum_nulo/antes*100:.3f}%)")

In [ ]:
mask_nulo = df_modelagem[colunas_nulos].isna().any(axis=1)

print("Distribuição de sigla_uf entre os alunos que seriam removidos:")
print(df_modelagem[mask_nulo]["sigla_uf"].value_counts())
print()
print("Distribuição do alvo entre os que seriam removidos:")
print(df_modelagem[mask_nulo]["alfabetizado"].value_counts(normalize=True).round(3) * 100)

Por ser uma porcentagem muito baixa de nulos, optamos por remover as linhas nulas do dataset

In [ ]:
df_modelagem = df_modelagem.dropna(subset=colunas_nulos)
print(df_modelagem.shape)
print()
print("Nulos restantes:")
nulos = df_modelagem.isna().sum()
print(nulos[nulos > 0] if (nulos > 0).any() else "Zero nulo restante")

In [ ]:
colunas_meta = ["meta_alfabetizacao_2024_municipio", "meta_alfabetizacao_2024_uf", "meta_alfabetizacao_2024_brasil"]

df_corr = df_alfabetizacao_presentes[colunas_meta].copy()
df_corr["alfabetizado_bin"] = (df_alfabetizacao_presentes["alfabetizado"] == "Sim").astype(int)

print("Correlação das metas com o alvo:")
print(df_corr.corr(numeric_only=True)["alfabetizado_bin"].drop("alfabetizado_bin").round(3))
print()
print("Correlação entre as 3 metas:")
print(df_corr[colunas_meta].corr().round(2))
print()
print("Valores distintos de meta_alfabetizacao_2024_brasil:", df_alfabetizacao_presentes["meta_alfabetizacao_2024_brasil"].nunique())

Podemos observar que meta_alfabetizacao_2024_brasil não tem nenhuma correlação, vamos remover do dataset

In [ ]:
df_modelagem = df_modelagem.drop(columns=["meta_alfabetizacao_2024_brasil"])
print(df_modelagem.shape)

100% dos alunos de rede Estadual ficam sem `meta_alfabetizacao_2024_municipio`. a meta municipal só existe pra rede Municipal. Descartar essas linhas eliminaria 10% da base inteira. Vamos preencher com a mediana e criar uma coluna indicadora, para o modelo distinguir "valor real" de "não se aplica".

In [ ]:
print(pd.crosstab(df_modelagem["rede"], df_modelagem["meta_alfabetizacao_2024_municipio"].isna()))

In [ ]:
df_modelagem["meta_alfabetizacao_2024_municipio_disponivel"] = df_modelagem["meta_alfabetizacao_2024_municipio"].notna().astype(int)

mediana_meta_municipio = df_modelagem["meta_alfabetizacao_2024_municipio"].median()
df_modelagem["meta_alfabetizacao_2024_municipio"] = df_modelagem["meta_alfabetizacao_2024_municipio"].fillna(mediana_meta_municipio)

print("Nulos restantes:", df_modelagem["meta_alfabetizacao_2024_municipio"].isna().sum())
print(df_modelagem["meta_alfabetizacao_2024_municipio_disponivel"].value_counts())

## Feature Encoding
transformando colunas textuais em numéricas

In [ ]:
# IDs
colunas_ids = ["id_aluno", "id_escola", "id_municipio", "id_municipio_nome"]

# constantes
colunas_variancia_zero = ["ano", "serie", "presenca", "_gold_processed_at"]

# duplicada (mesma informação de sigla_uf)
coluna_duplicada = ["sigla_uf_nome"]

# somente 249 casos de "prova não preenchida" contra 1.5 milhões de "preenchidas"
coluna_desbalanceada = ["preenchimento_caderno"]

df_modelagem = df_modelagem.drop(columns=colunas_ids + colunas_variancia_zero + coluna_duplicada + coluna_desbalanceada)
print(df_modelagem.shape)

### Utilizando one-hot encoding
O problema de utilizar outros encodings: </br>
label: criaria uma ordem, no nosso caso de UF não temos uma ordem. SP não é maior que BH por exemplo
target: poderiamos criar uma pista para o algoritmo com base na target

In [ ]:
df_modelagem = pd.get_dummies(df_modelagem, columns=["rede"], drop_first=True)
print(df_modelagem.shape)
print(df_modelagem.columns.tolist())

In [ ]:
df_modelagem = pd.get_dummies(df_modelagem, columns=["sigla_uf"], drop_first=True)
print(df_modelagem.shape)

In [ ]:
# checando as colunas finais para entrar no treino e teste
print(df_modelagem.columns.tolist())
df_modelagem.head()


In [ ]:
# definindo X e y
y = df_modelagem["alfabetizado_bin"]
X = df_modelagem.drop(columns=["alfabetizado", "alfabetizado_bin"])

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## Padronização

In [131]:
# não vamos adicionar as colunas de UF que mudamos no feature encoding
colunas_numericas = [
    "populacao", "pib_per_capita", "atu_ef_anos_iniciais", "dsu_ef_anos_iniciais",
    "afd_ef_anos_iniciais_grupo_1", "ird_alta", "ird_baixa_regularidade",
    "tdi_ef_2_ano_anterior", "taxa_aprovacao_ef_2_ano_anterior",
    "taxa_abandono_ef_2_ano_anterior", "inse_medio",
    "meta_alfabetizacao_2024_municipio", "meta_alfabetizacao_2024_uf",
]

scaler = StandardScaler()

scaler.fit(X_train[colunas_numericas])

X_train[colunas_numericas] = scaler.fit_transform(X_train[colunas_numericas])
X_test[colunas_numericas] = scaler.transform(X_test[colunas_numericas])

In [132]:
print("Média no treino (deve ser ~0):")
print(X_train[colunas_numericas].mean().round(3))
print()
print("Desvio padrão no treino (deve ser ~1):")
print(X_train[colunas_numericas].std().round(3))

Média no treino (deve ser ~0):
populacao                            0.0
pib_per_capita                       0.0
atu_ef_anos_iniciais                -0.0
dsu_ef_anos_iniciais                -0.0
afd_ef_anos_iniciais_grupo_1        -0.0
ird_alta                            -0.0
ird_baixa_regularidade               0.0
tdi_ef_2_ano_anterior                0.0
taxa_aprovacao_ef_2_ano_anterior    -0.0
taxa_abandono_ef_2_ano_anterior      0.0
inse_medio                          -0.0
meta_alfabetizacao_2024_municipio    0.0
meta_alfabetizacao_2024_uf          -0.0
dtype: float64

Desvio padrão no treino (deve ser ~1):
populacao                            1.0
pib_per_capita                       1.0
atu_ef_anos_iniciais                 1.0
dsu_ef_anos_iniciais                 1.0
afd_ef_anos_iniciais_grupo_1         1.0
ird_alta                             1.0
ird_baixa_regularidade               1.0
tdi_ef_2_ano_anterior                1.0
taxa_aprovacao_ef_2_ano_anterior     1.0
taxa_abandon